In [1]:
import streamlit as st
import pandas as pd
import os
import shutil
from langchain_community.vectorstores import Chroma # The database that stores vectors
from langchain_huggingface import HuggingFaceEmbeddings # The "translator" that turns text into numbers
from langchain_community.llms import Ollama # The interface to talk to Llama-3 locall

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all" 

c:\Users\irene\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# query_intent_sample

In [15]:
# get dataset ready
df = pd.read_csv("bank_customer_service_intent_classification_dataset.csv", encoding='latin1') 
df.head(1)

df = df[['query','intent']].drop_duplicates()
set(df.intent)

df = df[~df['intent'].isna()]
df = df[df['intent'].str.contains('_')]

df = df.groupby('intent').head(1).reset_index()
df


df = df.rename(columns={'query':'input','intent':'output'})
df

df_1 = df


# concat in trading related samples
examples_trade = [
            {"input": "I want to buy 100 shares of apple", "output": "trading"},
            {"input": "How much can I put in my 401k?", "output": "retirement"},
            {"input": "Where is my 1099 form?", "output": "tax"},
            {"input": "What is the weather like in Boston?", "output": "irrelevant"}
        ]

df = pd.DataFrame(examples_trade)
df.head(1)
df.shape

df_2 =df


df = pd.concat([df_1,df_2])
df.head(1)
df.shape

df.output.value_counts()
df.to_csv('./agent_memory_db/query_intent_sample.csv')

,Unnamed: 0,query,intent,Unnamed: 3,Unnamed: 4
0,0,Could you please help me reset my account pass...,password_reset,NaN,NaN


{' i heard u can appl 4 a credit card online',
 ' ur loan kinds',
 'balance_inquiry',
 'credi_card_application',
 'fraud_report',
 'loan_inquiry',
 nan,
 'password_reset',
 'transaction_query'}

,index,query,intent
0,0,Could you please help me reset my account pass...,password_reset
1,1,What company charged my account last?,transaction_query
2,2,How do I schedule an appointment to discuss lo...,loan_inquiry
3,6,yall better call me now,fraud_report
4,19,Can you provide instructions for uploading doc...,credi_card_application
5,58,I??e refreshed a million times??AVINGS BALANCE...,balance_inquiry


,index,input,output
0,0,Could you please help me reset my account pass...,password_reset
1,1,What company charged my account last?,transaction_query
2,2,How do I schedule an appointment to discuss lo...,loan_inquiry
3,6,yall better call me now,fraud_report
4,19,Can you provide instructions for uploading doc...,credi_card_application
5,58,I??e refreshed a million times??AVINGS BALANCE...,balance_inquiry


,input,output
0,I want to buy 100 shares of apple,trading


(4, 2)

,index,input,output
0,0.0,Could you please help me reset my account pass...,password_reset


(10, 3)

output
password_reset            1
transaction_query         1
loan_inquiry              1
fraud_report              1
credi_card_application    1
balance_inquiry           1
trading                   1
retirement                1
tax                       1
irrelevant                1
Name: count, dtype: int64

bulk_audit_sample.csv

In [2]:
bulk_audit_sample.csv

NameError: name 'bulk_audit_sample' is not defined

In [3]:
import pandas as pd # Import the pandas library for data management

# 1. Define the data as a list of dictionaries
data = [
    {"customer_statement": "I forgot my login password and need a reset", "department_routed": "password_reset", "confidence_level": 0.98},
    {"customer_statement": "Why was my $50 transfer to John Doe declined?", "department_routed": "transaction_query", "confidence_level": 0.45},
    {"customer_statement": "I want to apply for a new mortgage for a house", "department_routed": "loan_inquiry", "confidence_level": 0.89},
    {"customer_statement": "There is a weird charge on my card I didn't make", "department_routed": "fraud_report", "confidence_level": 0.32},
    {"customer_statement": "How do I get a new gold credit card?", "department_routed": "credi_card_application", "confidence_level": 0.55},
    {"customer_statement": "What is the current balance of my savings account?", "department_routed": "balance_inquiry", "confidence_level": 0.92},
    {"customer_statement": "I need to buy 50 shares of Apple stock", "department_routed": "trading", "confidence_level": 0.40},
    {"customer_statement": "Can I move money from my old 401k to a Roth IRA?", "department_routed": "retirement", "confidence_level": 0.95},
    {"customer_statement": "I am looking for my 1099-INT form for last year", "department_routed": "tax", "confidence_level": 0.38},
    {"customer_statement": "What is the best recipe for chocolate chip cookies?", "department_routed": "irrelevant", "confidence_level": 0.20},
    {"customer_statement": "I want to increase my credit limit on my existing card", "department_routed": "balance_inquiry", "confidence_level": 0.42},
    {"customer_statement": "Is the stock market open on Labor Day?", "department_routed": "trading", "confidence_level": 0.88}
]

# 2. Create the DataFrame
df = pd.DataFrame(data)

# 3. Save to CSV
# index=False ensures we don't add an extra column for the row numbers
df.to_csv("bulk_audit_sample.csv", index=False)

print("bulk_audit_sample.csv has been created using Pandas.")

bulk_audit_sample.csv has been created using Pandas.


# Tests

In [ ]:


# 1. Load your downloaded dataset
# We use pandas to read the local file you downloaded (e.g., Banking77 train.csv)
df = pd.read_csv("bank_customer_service_intent_classification_dataset.csv", encoding='latin1') 
df.head(1)

# 2. Slice the data (Optional)
# Banking77 is 10k rows; for a "small" local test, we might only take the first 500
small_df = df.head(500) 

# 3. Prepare the data for ChromaDB
# 'text' is the customer statement; 'label' is the intent name
# We convert the 'text' column into a Python list of strings
texts = small_df['query'].tolist() 

# We create a list of dictionaries for metadata so the LLM knows the "label" for each text
metadatas = [{"output": label} for label in small_df['intent'].tolist()] 

# 4. Initialize the Embeddings Model
# This turns each sentence into a mathematical vector (a list of numbers)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 5. Build the Vector Store from the CSV
# .from_texts() takes the lists, embeds them, and saves them to the 'persist_directory'
vectorstore = Chroma.from_texts(
    texts=texts,
    metadatas=metadatas,
    embedding=embeddings,
    persist_directory="./banking_memory_db" # The folder where the data is stored
)

print(f"Success! Loaded {len(texts)} real-world banking examples into ChromaDB.")

In [ ]:

class SemanticFidelityRouter:
    def __init__(self, db_path="./test_vector_db"):
        # Store the folder path where our "memory" will be saved on your hard drive
        self.db_path = db_path 
        
        # Load a pre-trained model that turns sentences into 384-dimensional vectors
        self.embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        
        # Initialize ChromaDB: it looks in 'db_path' for existing data or creates a new one
        self.vectorstore = Chroma(
            persist_directory=self.db_path,
            embedding_function=self.embeddings, # Tell Chroma how to "read" new text
            collection_name="intent_memory"     # A named "table" inside our database
        )
        
        # Connect to your local Ollama instance running Llama-3
        self.llm = Ollama(model="llama3")

# Create an instance of our class to use in the following cells
router = SemanticFidelityRouter()
print("Router initialized and Vector DB ready.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7956.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Success! Loaded 500 real-world banking examples into ChromaDB.


In [14]:
# The add_texts function performs three steps:
# 1. Runs the 'embeddings' model on the text
# 2. Pairs the resulting numbers with the metadata (the Department)
# 3. Writes it to the local 'test_vector_db' folder
router.vectorstore.add_texts(
    texts=["I need to download my 1099-R for my taxes"], # The sample sentence
    metadatas=[{"output": "Tax"}] # The "Ground Truth" label for the LLM to learn
)

router.vectorstore.add_texts(
    texts=["How do I buy shares of NVIDIA?"], 
    metadatas=[{"output": "Trading"}]
)

print("Memory updated with two specific examples.")

Memory updated with two specific examples.


In [15]:
# Access the raw collection object from the Chroma client
collection = router.vectorstore._collection

# Run a count operation on the database index
current_count = collection.count()

# Output the result: This should increase every time you run Cell 2
print(f"The system has {current_count} examples stored in its semantic memory.")

The system has 2 examples stored in its semantic memory.


In [17]:
# The user's new, unseen query
query = "I lost my login credentials"

# 'similarity_search' does the following:
# 1. Embeds the query into the same 384-dimensional space
# 2. Calculates the 'cosine distance' between this query and all stored examples
# 3. Returns the 'k' closest matches
results = router.vectorstore.similarity_search(query, k=1)
print(results)


# Loop through the results (even if k=1) to see what the system retrieved
for doc in results:
    # 'page_content' is the original text of the closest match
    print(f"Nearest Match Found: {doc.page_content}")
    # 'metadata' contains the department we assigned to that match
    print(f"Predicted Department: {doc.metadata['output']}")

[Document(metadata={'output': 'Tax'}, page_content='I need to download my 1099-R for my taxes')]
Nearest Match Found: I need to download my 1099-R for my taxes
Predicted Department: Tax


In [18]:
test_query = "Where is my tax document?"

# 1. Get the 2 most similar examples to this specific query
docs = router.vectorstore.similarity_search(test_query, k=2)

# 2. Format these examples into a text block (the 'Few-Shot' context)
context_block = ""
for d in docs:
    context_block += f"Input: {d.page_content} -> Label: {d.metadata['output']}\n"

# 3. Assemble the full string that goes to the LLM
final_prompt = f"""
Target Task: Classify the 'New Query' based on the 'Examples' provided.

Examples:
{context_block}

New Query: {test_query}
One-word Classification:"""

# Print the result so you can read the "cheat sheet" the LLM will receive
print(final_prompt)


Target Task: Classify the 'New Query' based on the 'Examples' provided.

Examples:
Input: I need to download my 1099-R for my taxes -> Label: Tax
Input: How do I buy shares of NVIDIA? -> Label: Trading


New Query: Where is my tax document?
One-word Classification:


In [20]:
# We take the prompt we just built and send it to the LLM
# This will take a few seconds depending on your GPU/CPU
response = router.llm.invoke(final_prompt)

# Use .strip() to remove any extra newlines or spaces the AI might return
cleaned_response = response.strip()

print(f"Llama-3 says this belongs to: {cleaned_response}")

Llama-3 says this belongs to: Based on the examples provided, I would classify the 'New Query' as:

Label: Tax


In [ ]:
# Check if the database folder exists
if os.path.exists("./test_vector_db"):
    # Delete the entire folder and all its contents (the embeddings and index)
    shutil.rmtree("./test_vector_db")
    print("Memory wiped. You are back to a blank slate.")
else:
    print("No database found to delete.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7984.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Success! Loaded 500 real-world banking examples into ChromaDB.
